# MERRY build
Rebuilds the MERRY auditor in `SNOWFLAKE_LEARNING_DB.MERRY`. The steps are: load the CSVs, run the rule views, build the contract search service and semantic view, create the agent, then validate.

**Stand-ins:** `EXTRACTED_*` tables are loaded from `truth_*.csv` with the `planted_error` column removed. `CONTRACT_CLAUSES` uses wording paraphrased from the README (`source='README_PLACEHOLDER'`). Once the PDFs are uploaded to `@DOCS`, replace both with the AI_EXTRACT / AI_PARSE_DOCUMENT cell at the end.

In [ ]:
%%sql -r setup_result
CREATE SCHEMA IF NOT EXISTS SNOWFLAKE_LEARNING_DB.MERRY;
USE SCHEMA SNOWFLAKE_LEARNING_DB.MERRY;
CREATE STAGE IF NOT EXISTS RAW_CSV;
CREATE STAGE IF NOT EXISTS DOCS ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE') DIRECTORY = (ENABLE = TRUE);
CREATE OR REPLACE FILE FORMAT CSV_FF TYPE=CSV PARSE_HEADER=TRUE FIELD_OPTIONALLY_ENCLOSED_BY='"' EMPTY_FIELD_AS_NULL=TRUE;
COPY FILES INTO @RAW_CSV FROM 'snow://workspace/SNOWFLAKE_LEARNING_DB.INVOICES.MERRY/versions/live/dataschema/' PATTERN='.*[.]csv';

In [ ]:
%%sql -r load_counts
USE SCHEMA SNOWFLAKE_LEARNING_DB.MERRY;
CREATE OR REPLACE TABLE RATE_CARD (contractor_code STRING, contractor_name STRING, rate_code STRING, item STRING, unit STRING, rate_php NUMBER(12,2), standby_rate_php NUMBER(12,2), term_start DATE, term_end DATE);
CREATE OR REPLACE TABLE GATE_LOG (gate_entry_id STRING, entry_date DATE, subject_type STRING, subject_id STRING, subject_name STRING, contractor_code STRING, gate STRING, time_in TIMESTAMP_NTZ, time_out TIMESTAMP_NTZ, remarks STRING);
CREATE OR REPLACE TABLE EQUIPMENT_LOG (log_date DATE, equipment_id STRING, contractor_code STRING, status STRING, hours_run NUMBER(6,2), remarks STRING);
CREATE OR REPLACE TABLE TIMESHEETS (work_date DATE, worker_id STRING, worker_name STRING, contractor_code STRING, role STRING, hours NUMBER(6,2), approved_by STRING);
CREATE OR REPLACE TABLE DAILY_REPORTS (report_date DATE, weather STRING, ops_status STRING, camp_headcount NUMBER, activities STRING, supervisor_notes STRING, reported_by STRING);
CREATE OR REPLACE TABLE TRUTH_INVOICE_LINES (invoice_no STRING, contractor_code STRING, week STRING, invoice_date DATE, period_start DATE, period_end DATE, line_no NUMBER, line_date STRING, ticket_ref STRING, description STRING, rate_code STRING, qty NUMBER(10,2), unit STRING, rate_php NUMBER(12,2), amount_php NUMBER(12,2), subtotal_php NUMBER(14,2), vat_php NUMBER(14,2), total_php NUMBER(14,2), planted_error STRING);
CREATE OR REPLACE TABLE TRUTH_TICKETS (ticket_no STRING, contractor_code STRING, ticket_date STRING, signed_by_company_rep BOOLEAN, handwritten BOOLEAN, scanned BOOLEAN, personnel STRING, equipment STRING, quantities STRING, remarks STRING);
CREATE OR REPLACE TABLE EVAL_ANSWER_KEY (error_id STRING, contractor_code STRING, week STRING, invoice_no STRING, description STRING, rule STRING, flagged_php NUMBER(12,2), decision STRING, correct_php NUMBER(12,2), overbilled_php NUMBER(12,2), withheld_php NUMBER(12,2), clauses STRING, evidence STRING);
CREATE OR REPLACE TABLE EVAL_QA_SAMPLES (tool STRING, question STRING, expected_answer STRING, source STRING);
COPY INTO RATE_CARD FROM @RAW_CSV/rate_card.csv FILE_FORMAT=(FORMAT_NAME=CSV_FF) MATCH_BY_COLUMN_NAME=CASE_INSENSITIVE;
COPY INTO GATE_LOG FROM @RAW_CSV/gate_log.csv FILE_FORMAT=(FORMAT_NAME=CSV_FF) MATCH_BY_COLUMN_NAME=CASE_INSENSITIVE;
COPY INTO EQUIPMENT_LOG FROM @RAW_CSV/equipment_log.csv FILE_FORMAT=(FORMAT_NAME=CSV_FF) MATCH_BY_COLUMN_NAME=CASE_INSENSITIVE;
COPY INTO TIMESHEETS FROM @RAW_CSV/timesheets.csv FILE_FORMAT=(FORMAT_NAME=CSV_FF) MATCH_BY_COLUMN_NAME=CASE_INSENSITIVE;
COPY INTO DAILY_REPORTS FROM @RAW_CSV/daily_reports.csv FILE_FORMAT=(FORMAT_NAME=CSV_FF) MATCH_BY_COLUMN_NAME=CASE_INSENSITIVE;
COPY INTO TRUTH_INVOICE_LINES FROM @RAW_CSV/truth_invoice_lines.csv FILE_FORMAT=(FORMAT_NAME=CSV_FF) MATCH_BY_COLUMN_NAME=CASE_INSENSITIVE;
COPY INTO TRUTH_TICKETS FROM @RAW_CSV/truth_tickets.csv FILE_FORMAT=(FORMAT_NAME=CSV_FF) MATCH_BY_COLUMN_NAME=CASE_INSENSITIVE;
COPY INTO EVAL_ANSWER_KEY FROM @RAW_CSV/answer_key.csv FILE_FORMAT=(FORMAT_NAME=CSV_FF) MATCH_BY_COLUMN_NAME=CASE_INSENSITIVE;
COPY INTO EVAL_QA_SAMPLES FROM @RAW_CSV/qa_samples.csv FILE_FORMAT=(FORMAT_NAME=CSV_FF) MATCH_BY_COLUMN_NAME=CASE_INSENSITIVE;
-- Stand-ins for AI_EXTRACT output (planted_error dropped)
CREATE OR REPLACE TABLE EXTRACTED_INVOICE_LINES AS SELECT invoice_no, contractor_code, week, invoice_date, period_start, period_end, line_no, TRY_TO_DATE(line_date) line_date, ticket_ref, description, rate_code, qty, unit, rate_php, amount_php, subtotal_php, vat_php, total_php, 'CSV_STANDIN' source FROM TRUTH_INVOICE_LINES;
CREATE OR REPLACE TABLE EXTRACTED_TICKETS AS SELECT ticket_no, contractor_code, TRY_TO_DATE(ticket_date) ticket_date, signed_by_company_rep, handwritten, scanned, personnel, equipment, quantities, remarks, 'CSV_STANDIN' source FROM TRUTH_TICKETS;
SELECT 'gate_log' t, COUNT(*) n FROM GATE_LOG UNION ALL SELECT 'invoice_lines', COUNT(*) FROM EXTRACTED_INVOICE_LINES UNION ALL SELECT 'tickets', COUNT(*) FROM EXTRACTED_TICKETS;

In [ ]:
%%sql -r rules_result
USE SCHEMA SNOWFLAKE_LEARNING_DB.MERRY;
CREATE OR REPLACE VIEW R_RATE AS
WITH l AS (
  SELECT i.*, TRIM(SPLIT_PART(description,' - ',1)) billed_role,
         TRIM(REGEXP_SUBSTR(SPLIT_PART(SPLIT_PART(description,' - ',2),',',1),'[A-Za-z]+$')) surname
  FROM EXTRACTED_INVOICE_LINES i WHERE description LIKE '% - %')
SELECT l.invoice_no, l.contractor_code, l.week, 'rate' rule, l.line_no, l.line_date, l.ticket_ref,
  l.description || ': billed as ' || l.billed_role || ' @' || l.rate_php || ', timesheet role ' || t.role || ' @' || rc.rate_php detail,
  (l.rate_php - rc.rate_php) * l.qty flagged_php,
  'timesheet ' || t.worker_id || ' role=' || t.role || '; rate_card ' || rc.rate_code || ' ' || rc.rate_php evidence
FROM l
JOIN TIMESHEETS t ON t.contractor_code=l.contractor_code AND t.work_date=l.line_date AND t.worker_name ILIKE '%' || l.surname
JOIN RATE_CARD rc ON rc.contractor_code=l.contractor_code AND rc.item ILIKE t.role
WHERE t.role NOT ILIKE l.billed_role AND l.rate_php > rc.rate_php;

CREATE OR REPLACE VIEW R_MATH AS
SELECT invoice_no, contractor_code, week, 'math' rule, line_no, line_date, ticket_ref,
  description || ': ' || qty || ' x ' || rate_php || ' = ' || qty*rate_php || ' but shown as ' || amount_php detail,
  amount_php - qty*rate_php flagged_php, 'invoice line ' || COALESCE(rate_code,'') evidence
FROM EXTRACTED_INVOICE_LINES WHERE ABS(amount_php - qty*rate_php) > 1;

CREATE OR REPLACE VIEW R_DUPLICATES AS
SELECT l.invoice_no, l.contractor_code, l.week, 'duplicates' rule, l.line_no, l.line_date, l.ticket_ref,
  'Ticket ' || l.ticket_ref || ' already billed on ' || p.invoice_no detail, l.amount_php flagged_php,
  'invoice ' || p.invoice_no || ' ticket ' || l.ticket_ref evidence
FROM EXTRACTED_INVOICE_LINES l
JOIN (SELECT DISTINCT invoice_no, invoice_date, ticket_ref, contractor_code FROM EXTRACTED_INVOICE_LINES) p
  ON p.ticket_ref=l.ticket_ref AND p.contractor_code=l.contractor_code AND p.invoice_date < l.invoice_date;

CREATE OR REPLACE VIEW R_TICKET_BACKING AS
SELECT l.invoice_no, l.contractor_code, l.week, 'ticket_backing' rule, l.line_no, l.line_date, l.ticket_ref,
  l.description || ': ticket ' || l.ticket_ref || CASE WHEN t.ticket_no IS NULL THEN ' not found' ELSE ' not signed by Company Rep' END detail,
  l.amount_php flagged_php, 'ticket ' || l.ticket_ref || ' signed_by_company_rep=' || COALESCE(t.signed_by_company_rep::STRING,'missing') evidence
FROM EXTRACTED_INVOICE_LINES l
LEFT JOIN EXTRACTED_TICKETS t ON t.ticket_no=l.ticket_ref
WHERE l.ticket_ref IS NOT NULL AND l.ticket_ref NOT ILIKE '% to %'
  AND (t.ticket_no IS NULL OR t.signed_by_company_rep = FALSE);

CREATE OR REPLACE VIEW R_ELIGIBILITY AS
SELECT l.invoice_no, l.contractor_code, l.week, 'eligibility' rule, l.line_no, l.line_date, l.ticket_ref,
  CASE WHEN rc.rate_code IS NULL THEN l.description || ': not a contract rate item'
       ELSE l.description || ' on ' || l.line_date || ': outside term ' || rc.term_start || '..' || rc.term_end END detail,
  l.amount_php flagged_php,
  CASE WHEN rc.rate_code IS NULL THEN 'rate_card has no item for this line' ELSE 'rate_card ' || rc.rate_code || ' term_end ' || rc.term_end END evidence
FROM EXTRACTED_INVOICE_LINES l
LEFT JOIN RATE_CARD rc ON rc.rate_code=l.rate_code
WHERE rc.rate_code IS NULL OR l.line_date NOT BETWEEN rc.term_start AND rc.term_end;

CREATE OR REPLACE VIEW R_SITE_RECORD_MATCH AS
-- equipment days billed vs OPERATING days in equipment_log
SELECT l.invoice_no, l.contractor_code, l.week, 'site_record_match' rule, MIN(l.line_no) line_no, NULL::DATE line_date, ANY_VALUE(l.ticket_ref) ticket_ref,
  ANY_VALUE(rc.item) || ': billed ' || SUM(l.qty) || ' days, equipment_log shows ' || ANY_VALUE(e.op_days) || ' OPERATING; non-operating: ' || COALESCE(ANY_VALUE(e.other_days),'none') detail,
  (SUM(l.qty) - ANY_VALUE(e.op_days)) * ANY_VALUE(l.rate_php) flagged_php,
  'equipment_log ' || ANY_VALUE(e.equipment_id) || ' ' || COALESCE(ANY_VALUE(e.other_days),'') evidence
FROM EXTRACTED_INVOICE_LINES l
JOIN RATE_CARD rc ON rc.rate_code=l.rate_code AND rc.rate_code LIKE '%-EQ-%'
JOIN LATERAL (
  SELECT ANY_VALUE(el.equipment_id) equipment_id, COUNT_IF(el.status='OPERATING') op_days,
         LISTAGG(IFF(el.status<>'OPERATING', el.log_date || ' ' || el.status, NULL), ', ') other_days
  FROM EQUIPMENT_LOG el
  WHERE el.equipment_id = REGEXP_SUBSTR(rc.item,'\\(([^)]+)\\)',1,1,'e',1)
    AND el.log_date BETWEEN l.period_start AND l.period_end) e
GROUP BY l.invoice_no, l.contractor_code, l.week, l.rate_code
HAVING SUM(l.qty) > ANY_VALUE(e.op_days)
UNION ALL
-- hourly labour billed vs gate hours
SELECT x.invoice_no, x.contractor_code, x.week, 'site_record_match', MIN(x.line_no), x.line_date, ANY_VALUE(x.ticket_ref),
  'Hours billed ' || SUM(x.qty) || ' vs ' || SUM(x.gate_hrs) || ' on site at gate (' || LISTAGG(DISTINCT x.gate, '/') || ')',
  SUM((x.qty - x.gate_hrs) * x.rate_php),
  'gate_log ' || x.contractor_code || ' ' || x.line_date || ' ' || LISTAGG(DISTINCT x.gate, '/')
FROM (
  SELECT l.invoice_no, l.contractor_code, l.week, l.line_no, l.line_date, l.ticket_ref, l.qty, l.rate_php,
         ROUND(SUM(DATEDIFF('minute', g.time_in, g.time_out))/60, 1) gate_hrs, ANY_VALUE(g.gate) gate
  FROM EXTRACTED_INVOICE_LINES l
  JOIN RATE_CARD rc ON rc.rate_code=l.rate_code AND rc.unit='hour'
  JOIN TIMESHEETS t ON t.contractor_code=l.contractor_code AND t.work_date=l.line_date AND t.role=rc.item
  JOIN GATE_LOG g ON g.subject_id=t.worker_id AND g.entry_date=l.line_date
  GROUP BY 1,2,3,4,5,6,7,8) x
WHERE x.qty - x.gate_hrs > 0.5
GROUP BY x.invoice_no, x.contractor_code, x.week, x.line_date;

CREATE OR REPLACE VIEW V_EXCEPTION_LINES AS
SELECT * FROM R_RATE UNION ALL SELECT * FROM R_MATH UNION ALL SELECT * FROM R_DUPLICATES
UNION ALL SELECT * FROM R_TICKET_BACKING UNION ALL SELECT * FROM R_ELIGIBILITY UNION ALL SELECT * FROM R_SITE_RECORD_MATCH;

CREATE OR REPLACE VIEW V_EXCEPTIONS AS
SELECT 'X-' || invoice_no || '-' || UPPER(rule) exception_id, invoice_no, contractor_code, week, rule,
  SUM(flagged_php) flagged_php, LISTAGG(detail, ' | ') details, LISTAGG(DISTINCT evidence, '; ') evidence
FROM V_EXCEPTION_LINES GROUP BY invoice_no, contractor_code, week, rule;

In [ ]:
%%sql -r rule_check
SELECT a.error_id, COALESCE(e.invoice_no, a.invoice_no) invoice_no, COALESCE(e.rule, a.rule) rule,
  e.flagged_php merry_flagged, a.flagged_php expected_flagged,
  IFF(e.flagged_php = a.flagged_php, 'MATCH', 'MISMATCH') status
FROM SNOWFLAKE_LEARNING_DB.MERRY.V_EXCEPTIONS e
FULL OUTER JOIN SNOWFLAKE_LEARNING_DB.MERRY.EVAL_ANSWER_KEY a ON a.invoice_no = e.invoice_no AND a.rule = e.rule
ORDER BY 1;

In [ ]:
%%sql -r search_result
USE SCHEMA SNOWFLAKE_LEARNING_DB.MERRY;
-- Parse contract_msa.pdf from the workspace -> clause chunks -> Cortex Search
COPY FILES INTO @DOCS FROM 'snow://workspace/SNOWFLAKE_LEARNING_DB.INVOICES.MERRY/versions/live/' FILES=('contract_msa.pdf');
ALTER STAGE DOCS REFRESH;
CREATE OR REPLACE TABLE CONTRACT_PARSED AS
SELECT AI_PARSE_DOCUMENT(TO_FILE('@DOCS','contract_msa.pdf'), {'mode':'LAYOUT'}):content::STRING AS doc_text;

CREATE OR REPLACE TABLE CONTRACT_CLAUSES AS
WITH clean AS (
  SELECT REGEXP_REPLACE(REGEXP_REPLACE(doc_text, 'Master Service Agreement No\\. PUP-MSA-2026-014 \\| Confidential\\s*|Page \\d+\\s*', ''), '&amp;', '&') t FROM CONTRACT_PARSED),
marked AS (
  SELECT REGEXP_REPLACE(t, '\n(\\d+\\.\\d+ |#+ ARTICLE|# EXHIBIT|## A\\.\\d)', '\n~~~\\1') t FROM clean),
chunks AS (
  SELECT s.index idx, TRIM(s.value) chunk FROM marked, LATERAL SPLIT_TO_TABLE(marked.t, '~~~') s)
SELECT
  COALESCE(REGEXP_SUBSTR(chunk, '^(\\d+\\.\\d+) ', 1, 1, 'e'),
           REGEXP_SUBSTR(chunk, '^## (A\\.\\d)', 1, 1, 'e'),
           IFF(chunk ILIKE '# EXHIBIT B%', 'EXHIBIT B', NULL),
           IFF(chunk ILIKE '# EXHIBIT A%', 'EXHIBIT A', NULL),
           IFF(idx = 1, 'PREAMBLE', NULL)) clause_id,
  LAST_VALUE(REGEXP_SUBSTR(chunk, '^#+ (ARTICLE \\d+\\. [A-Z ,&]+|EXHIBIT [AB]: [A-Z ]+)', 1, 1, 'e') IGNORE NULLS)
     OVER (ORDER BY idx ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) article,
  COALESCE(REGEXP_SUBSTR(chunk, '^\\d+\\.\\d+ ([^.]+)\\.', 1, 1, 'e'), REGEXP_SUBSTR(chunk, '^#+ ([^\n]+)', 1, 1, 'e')) title,
  chunk clause_text,
  'contract_msa.pdf' source
FROM chunks
WHERE chunk NOT RLIKE '^#+ ARTICLE[^\n]*$' AND LENGTH(chunk) > 20;
DELETE FROM CONTRACT_CLAUSES WHERE clause_id IS NULL;
UPDATE CONTRACT_CLAUSES SET article = 'ARTICLE 2. TERM' WHERE clause_id LIKE '2.%';
UPDATE CONTRACT_CLAUSES SET clause_text = TRIM(REGEXP_REPLACE(clause_text, '\\s*#+ ARTICLE \\d+\\.[^\n]*\\s*$', ''));
UPDATE CONTRACT_CLAUSES SET article = 'PREAMBLE' WHERE clause_id = 'PREAMBLE';

CREATE OR REPLACE CORTEX SEARCH SERVICE CONTRACT_SEARCH
  ON clause_text ATTRIBUTES clause_id, title
  WAREHOUSE = SNOWFLAKE_LEARNING_WH TARGET_LAG = '1 day'
  AS SELECT clause_id, article, title, clause_text, source FROM CONTRACT_CLAUSES;
SELECT clause_id, article, title FROM CONTRACT_CLAUSES ORDER BY clause_id;

In [ ]:
%%sql -r sv_result
CREATE OR REPLACE SEMANTIC VIEW SNOWFLAKE_LEARNING_DB.MERRY.SITE_RECORDS_SV
TABLES (
  rate_card AS SNOWFLAKE_LEARNING_DB.MERRY.RATE_CARD PRIMARY KEY (rate_code) COMMENT='Contract rates (Exhibit A) and contractor terms (Exhibit B). Codes with -EQ- are equipment; standby_rate_php is 50% of day rate.',
  gate_log AS SNOWFLAKE_LEARNING_DB.MERRY.GATE_LOG PRIMARY KEY (gate_entry_id) COMMENT='Every entry/exit through site gates at Well Pad BK-07. PERSON rows are one per worker per day; each water truck trip is its own EQUIPMENT row (trips = loads).',
  equipment_log AS SNOWFLAKE_LEARNING_DB.MERRY.EQUIPMENT_LOG COMMENT='Daily status of PU-101 pump, CR-301 crane, WL-501 wireline unit. IDLE = on location unused due to weather (standby-billable); OFF_HIRE = released from paid hire; DEMOBILIZED.',
  timesheets AS SNOWFLAKE_LEARNING_DB.MERRY.TIMESHEETS COMMENT='One row per worker per day worked, with role and hours (may include travel).',
  daily_reports AS SNOWFLAKE_LEARNING_DB.MERRY.DAILY_REPORTS PRIMARY KEY (report_date) COMMENT='Company daily report: weather, ops_status (NORMAL/SUSPENDED), camp headcount, activities, supervisor notes.',
  invoice_lines AS SNOWFLAKE_LEARNING_DB.MERRY.EXTRACTED_INVOICE_LINES COMMENT='Contractor invoice line items (pre-VAT) extracted from invoice PDFs.',
  exceptions AS SNOWFLAKE_LEARNING_DB.MERRY.V_EXCEPTIONS PRIMARY KEY (exception_id) COMMENT='Audit exceptions flagged by SQL rules (rate, math, duplicates, ticket_backing, eligibility, site_record_match), one per invoice per rule.'
)
RELATIONSHIPS (
  invoice_rate AS invoice_lines (rate_code) REFERENCES rate_card
)
FACTS (
  gate_log.hours_on_site AS ROUND(DATEDIFF('minute', time_in, time_out)/60, 1) COMMENT='Hours between gate entry and exit',
  equipment_log.hours_run_f AS hours_run COMMENT='Engine hours that day',
  timesheets.hours_f AS hours COMMENT='Timesheet hours',
  invoice_lines.amount AS amount_php COMMENT='Line amount PHP',
  invoice_lines.quantity AS qty,
  exceptions.flagged AS flagged_php
)
DIMENSIONS (
  rate_card.rc_contractor_code AS contractor_code, rate_card.contractor_name AS contractor_name,
  rate_card.rate_code AS rate_code, rate_card.item AS item, rate_card.unit AS unit,
  rate_card.rate_php AS rate_php, rate_card.standby_rate_php AS standby_rate_php,
  rate_card.term_start AS term_start, rate_card.term_end AS term_end,
  gate_log.entry_date AS entry_date, gate_log.subject_type AS subject_type, gate_log.subject_id AS subject_id,
  gate_log.subject_name AS subject_name, gate_log.gate_contractor_code AS contractor_code, gate_log.gate AS gate,
  gate_log.time_in AS time_in, gate_log.time_out AS time_out, gate_log.gate_remarks AS remarks,
  equipment_log.log_date AS log_date, equipment_log.equipment_id AS equipment_id, equipment_log.eq_contractor_code AS contractor_code,
  equipment_log.status AS status WITH SYNONYMS=('equipment status') COMMENT='OPERATING, IDLE, OFF_HIRE, DEMOBILIZED',
  equipment_log.eq_remarks AS remarks,
  timesheets.work_date AS work_date, timesheets.worker_id AS worker_id, timesheets.worker_name AS worker_name,
  timesheets.ts_contractor_code AS contractor_code, timesheets.role AS role, timesheets.approved_by AS approved_by,
  daily_reports.report_date AS report_date, daily_reports.weather AS weather, daily_reports.ops_status AS ops_status,
  daily_reports.camp_headcount AS camp_headcount, daily_reports.activities AS activities, daily_reports.supervisor_notes AS supervisor_notes,
  invoice_lines.invoice_no AS invoice_no, invoice_lines.inv_contractor_code AS contractor_code, invoice_lines.week AS week,
  invoice_lines.invoice_date AS invoice_date, invoice_lines.line_date AS line_date, invoice_lines.ticket_ref AS ticket_ref,
  invoice_lines.description AS description, invoice_lines.inv_rate_code AS rate_code, invoice_lines.line_rate_php AS rate_php,
  exceptions.exception_id AS exception_id, exceptions.exc_invoice_no AS invoice_no, exceptions.exc_contractor_code AS contractor_code,
  exceptions.rule AS rule, exceptions.details AS details, exceptions.evidence AS evidence
)
METRICS (
  gate_log.total_hours_on_site AS SUM(gate_log.hours_on_site),
  gate_log.gate_entries AS COUNT(gate_entry_id) COMMENT='Number of gate entries; for AQH water trucks equals loads delivered',
  equipment_log.days_logged AS COUNT(log_date),
  timesheets.total_timesheet_hours AS SUM(timesheets.hours_f),
  invoice_lines.total_billed AS SUM(invoice_lines.amount),
  exceptions.total_flagged AS SUM(exceptions.flagged)
)
COMMENT='MERRY site records for Well Pad BK-07 (Pinnacle Upstream, MSA PUP-MSA-2026-014), Aug 31 - Sep 27 2026. Contractors: PCS PumpCo, LCR Liftmax, AQH AquaHaul, KCS Kusina, TLW TruLine.';

In [ ]:
%%sql -r agent_result
CREATE OR REPLACE AGENT SNOWFLAKE_LEARNING_DB.MERRY.MERRY_AGENT
COMMENT = 'MERRY: agentic auditor for contractor invoices and field tickets, Well Pad BK-07'
FROM SPECIFICATION $$
models:
  orchestration: auto
orchestration:
  budget:
    seconds: 300
    tokens: 400000
instructions:
  system: >
    You are MERRY, the invoice auditor for Pinnacle Upstream Philippines Inc., Well Pad BK-07, under master service
    agreement PUP-MSA-2026-014. Company representative: Ramon Dizon. Contractors: PCS PumpCo Services, LCR Liftmax Crane & Rigging,
    AQH AquaHaul Trucking, KCS Kusina Camp Services, TLW TruLine Wireline Services. Currency is PHP (show as ₱); all amounts are pre-VAT.
  orchestration: >
    Audit workflow for each exception:
    1. Get the rule-flagged exceptions with the site_records tool (exceptions table: exception_id, invoice_no, rule, flagged_php, details, evidence).
    2. Search the contract with contract_search for the clause that governs the issue
    (2.1 term, 4.1 rates incl. fuel, 4.2 standby 50% for idle equipment and stood-down crew not billable, 5.1 signed tickets,
    5.3 hours gate-to-gate, 5.4 travel time up to 2 h each way, 6.1 ticket billed once, 7.3 exclusions).
    3. Verify the facts with site_records (equipment_log status, gate_log hours/gate, timesheets roles, daily_reports weather/ops_status/supervisor_notes).
    4. Decide one of:
       CONFIRM - the full flagged amount is overbilled (or, for an unsigned ticket under 5.1, WITHHELD until signed, not overbilled);
       CORRECT - part of the flagged amount is legitimately billable, e.g. equipment IDLE on location during weather is billable at the
       standby rate (50% of day rate), so the overbilled amount = flagged minus standby value;
       DISMISS - the contract allows the charge, e.g. hours above gate time that are covered by allowed travel time (clause 5.4,
       at most 2 hours each way, supported by supervisor notes about travel from the yard).
    5. Report per exception: decision, flagged ₱, correct/overbilled ₱, withheld ₱, clause cited (number and wording), and the evidence rows.
    When asked, draft a concise, professional dispute letter to the contractor listing each upheld item, clause and evidence, and the amount to credit or withhold.
    Point out repeat patterns across weeks (for example the same contractor billing standby days at full rate more than once).
    Never invent rates, dates or clauses; if evidence is missing, say so.
  response: >
    Be concise. Use a table for multiple exceptions with columns: ID, invoice, rule, flagged ₱, decision, overbilled ₱, withheld ₱, clause.
    Give totals at the end.
  sample_questions:
    - question: "Audit PumpCo invoice PCS-INV-1042"
    - question: "Review all flagged exceptions and give me the totals"
    - question: "Is the TruLine hours exception on TLW-INV-5513 valid?"
    - question: "Draft the dispute letter to PumpCo"
tools:
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: site_records
      description: >
        Query structured audit data: site records (gate_log, equipment_log, timesheets, daily_reports), the rate card and contract terms,
        extracted invoice lines, and the rule-flagged audit exceptions. Use for any facts, counts, hours, statuses, rates and amounts.
  - tool_spec:
      type: cortex_search
      name: contract_search
      description: Search the master service agreement clauses (terms, rates, standby, field tickets, hours, travel time, invoicing, exclusions, Exhibits A and B). Use to find and cite the governing clause.
tool_resources:
  site_records:
    semantic_view: SNOWFLAKE_LEARNING_DB.MERRY.SITE_RECORDS_SV
    execution_environment:
      type: warehouse
      warehouse: SNOWFLAKE_LEARNING_WH
  contract_search:
    search_service: SNOWFLAKE_LEARNING_DB.MERRY.CONTRACT_SEARCH
    id_column: CLAUSE_ID
    title_column: TITLE
    max_results: 4
$$;

## 8. When the PDFs are available (template, not run)
Upload `contract_msa.pdf`, `invoices/*.pdf` and `tickets/*.pdf` to `@SNOWFLAKE_LEARNING_DB.MERRY.DOCS`. Then replace the stand-in tables:

```sql
ALTER STAGE SNOWFLAKE_LEARNING_DB.MERRY.DOCS REFRESH;

-- Contract -> clause chunks for Cortex Search
CREATE OR REPLACE TABLE CONTRACT_PARSED AS
SELECT AI_PARSE_DOCUMENT(TO_FILE('@DOCS','contract_msa.pdf'), {'mode':'LAYOUT'}):content::STRING AS doc_text;
-- split doc_text into one row per clause (e.g. REGEXP on '\n\d\.\d ') and INSERT INTO CONTRACT_CLAUSES, then recreate CONTRACT_SEARCH

-- Invoices -> header + line items
CREATE OR REPLACE TABLE INVOICE_EXTRACT_RAW AS
SELECT relative_path,
  AI_EXTRACT(file => TO_FILE('@DOCS', relative_path),
    responseFormat => {'schema': {'type':'object','properties':{
      'invoice_no':{'type':'string'},'invoice_date':{'type':'string'},'period_start':{'type':'string'},'period_end':{'type':'string'},
      'lines':{'type':'object','column_ordering':['line_date','ticket_ref','description','rate_code','qty','unit','rate_php','amount_php'],
        'properties':{'line_date':{'type':'array'},'ticket_ref':{'type':'array'},'description':{'type':'array'},'rate_code':{'type':'array'},
                      'qty':{'type':'array'},'unit':{'type':'array'},'rate_php':{'type':'array'},'amount_php':{'type':'array'}}}}}}) AS j
FROM DIRECTORY(@DOCS) WHERE relative_path ILIKE 'invoices/%';

-- Tickets -> signature + content
CREATE OR REPLACE TABLE TICKET_EXTRACT_RAW AS
SELECT relative_path,
  AI_EXTRACT(file => TO_FILE('@DOCS', relative_path), responseFormat => {
    'ticket_no':'Field ticket number','ticket_date':'Ticket date (YYYY-MM-DD)',
    'company_rep_signed':'Is the Company Representative signature present? yes or no',
    'personnel':'Personnel names, roles and hours','equipment':'Equipment and days','quantities':'Quantities delivered or handled'}) AS j
FROM DIRECTORY(@DOCS) WHERE relative_path ILIKE 'tickets/%';
-- Flatten into EXTRACTED_INVOICE_LINES / EXTRACTED_TICKETS (same columns), rerun cell 4 to validate against the answer key,
-- and compare to TRUTH_INVOICE_LINES / TRUTH_TICKETS to measure extraction accuracy.
```